In [ ]:
from src.autotune.tune import (
    tune_time_to_optimal
)
from src.autotune.models import SolverFactory, ModelSolver, SolutionStatus
from src.cpsat_autotune.cp_sat_solver import CpSatSolverFactory, CpSatSolver


In [ ]:
# ImportError: attempted relative import beyond top-level package
import ortools
ortools.__version__

In [ ]:
from ortools.sat.python.cp_model import CpModel, CpSolver
model1 = CpModel()
model2 = CpModel()
models = [model1, model2]

In [ ]:
class ArrayModelSolver(CpSatSolver):

    def solve(self, model : list[CpModel])->SolutionStatus:
        for m in model:
            self.solver.solve(m)
        return SolutionStatus.OPTIMAL

In [ ]:
class ArrayFactory(CpSatSolverFactory):
    def prepare_solver(self, params: dict[str, float | int | bool | list | tuple]) -> ModelSolver[list[CpModel]]:
        return ArrayModelSolver(solver=self._prepare_cp_sat_solver(params))

In [ ]:
from src.autotune.parameter_space import ParameterSpace
from src.autotune import tune
from src.cpsat_autotune.cp_sat_parameters import CPSAT_PARAMETER_SUGGESTIONS, CPSAT_PARAMETERS

In [ ]:
# !pip uninstall ortools

In [ ]:
parameter_space = ParameterSpace(
    all_parameters=CPSAT_PARAMETERS,
    parameters=CPSAT_PARAMETER_SUGGESTIONS
)
parameter_space.drop_parameter("use_lns_only")  # Not useful for this metric
parameter_space.drop_parameter("max_time_in_seconds")
parameter_space.filter_applicable_parameters(models)

In [ ]:
result = tune_time_to_optimal(
     model = models,
    solver_factory = ArrayFactory(),
    parameter_space = parameter_space,
    max_time_in_seconds = 10,
    relative_gap_limit= 0.0,
    n_samples_for_trial = 10,
    n_samples_for_verification = 30,
    n_trials= 100,
)
result